### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [19]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, AsyncOpenAI
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [20]:
from accounts import Account

In [30]:
account = Account.get("Amith")
account

Account(name='amith', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [31]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "amith", "balance": 9921.844, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 26.052, "timestamp": "2026-04-21 11:43:25", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-04-21 11:43:25", 10125.844]], "total_portfolio_value": 10125.844, "total_profit_loss": 125.84399999999914}'

In [32]:
account.report()

'{"name": "amith", "balance": 9921.844, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 26.052, "timestamp": "2026-04-21 11:43:25", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-04-21 11:43:25", 10125.844], ["2026-04-21 11:43:29", 10032.844]], "total_portfolio_value": 10032.844, "total_profit_loss": 32.84399999999914}'

In [33]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 26.052,
  'timestamp': '2026-04-21 11:43:25',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [35]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [36]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [ ]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Amith and my account is under the name Amith. What's my balance and my holdings?"
model_name = "llama3.2"

In [28]:
import os
import httpx

# WSL users can reach a Windows-hosted Ollama at host.docker.internal.
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

async def verify_ollama(base_url: str) -> str:
    candidates = [base_url]
    if "localhost" in base_url:
        candidates.append(base_url.replace("localhost", "host.docker.internal"))

    for candidate in candidates:
        health_url = candidate.removesuffix("/v1") + "/api/tags"
        try:
            async with httpx.AsyncClient(timeout=3) as client:
                response = await client.get(health_url)
                if response.status_code == 200:
                    return candidate
        except Exception:
            pass

    raise RuntimeError(
        "Could not reach Ollama. Start it with `ollama serve`, then ensure "
        "`ollama pull llama3.2` has completed. If Ollama runs on Windows, set "
        "OLLAMA_BASE_URL=http://host.docker.internal:11434/v1 in WSL."
    )

working_base_url = await verify_ollama(OLLAMA_BASE_URL)
print(f"Using Ollama endpoint: {working_base_url}")

ollama_client = AsyncOpenAI(base_url=working_base_url, api_key="ollama")
model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)

Using Ollama endpoint: http://host.docker.internal:11434/v1


In [29]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=240) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, your current balance is $9,540.82. 

Your holdings consist of 6 shares of Amazon (AMZN).

### Now let's build our own MCP Client

In [ ]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

In [ ]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
context = await read_accounts_resource("ed")
print(context)

In [ ]:
from accounts import Account
Account.get("ed").report()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>